# Johnson & Johnson Part 2 Step A: Signal Computation

**Author:** Roshan Patel  
**Course:** 15.465 Alphanomics, Spring 2026  
**Cutoff date:** 30 April 2026  
**Reporting basis:** USD (US GAAP).

This notebook produces every value in the Step A signal table. Each section is independently runnable. Data caches live in `data/` so re-running does not re-fetch.

Dependencies: `pandas`, `numpy`, `requests`, `yfinance`. WRDS endpoints require the `WRDS_API_TOKEN` environment variable to be set.

## Setup

In [1]:
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import yfinance as yf

DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

WRDS_TOKEN = os.environ.get('WRDS_API_TOKEN')
if not WRDS_TOKEN:
    raise RuntimeError(
        'Set WRDS_API_TOKEN before running. In a shell: '
        'export WRDS_API_TOKEN=your_token_here'
    )

WRDS_BASE = 'https://wrds-api.wharton.upenn.edu/data'

/Users/Roshan/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. Reference dates

JNJ Q4 2025 results were released on 21 January 2026. The 10-K for fiscal year ending 28 December 2025 was filed with the SEC on 11 February 2026. The team is using end of April 2026 as the project cutoff. Q1 2026 results are scheduled for late April 2026 but had not been released by the cutoff.

In [2]:
CUTOFF = pd.Timestamp('2026-04-30')
Q4_2025_ANNOUNCEMENT = pd.Timestamp('2026-01-21')
MOMENTUM_WINDOW_END = pd.Timestamp('2026-03-31')   # skip most recent month
MOMENTUM_WINDOW_START = pd.Timestamp('2025-03-31')
VOL_WINDOW_END = CUTOFF
VOL_WINDOW_START = CUTOFF - pd.Timedelta(days=365)
INSIDER_WINDOW_START = CUTOFF - pd.DateOffset(months=3)

print('Project cutoff:           ', CUTOFF.date())
print('Q4 2025 announcement:     ', Q4_2025_ANNOUNCEMENT.date())
print('Momentum window:          ', MOMENTUM_WINDOW_START.date(), '->', MOMENTUM_WINDOW_END.date())
print('Volatility window:        ', VOL_WINDOW_START.date(), '->', VOL_WINDOW_END.date())
print('Insider 3-month window:   ', INSIDER_WINDOW_START.date(), '->', CUTOFF.date())

Project cutoff:            2026-04-30
Q4 2025 announcement:      2026-01-21
Momentum window:           2025-03-31 -> 2026-03-31
Volatility window:         2025-04-30 -> 2026-04-30
Insider 3-month window:    2026-01-30 -> 2026-04-30


## 2. WRDS IBES data

JNJ has full IBES coverage as a US-domestic issuer. Pulls:

1. Quarterly EPS actuals through Q4 2025 (`ibes.actu_epsus`).
2. FY 2025 annual consensus history (`ibes.statsum_epsus`); the most recent statpers before the 21 January 2026 announcement is the relevant snapshot.

IBES tracks the 'street' (adjusted) EPS for major US issuers, which is what analysts forecast. JNJ Q4 2025 IBES actual is $2.46 (adjusted EPS), versus the $2.10 reported GAAP EPS that includes a $0.10 charge for the Halda Therapeutics acquisition. Using the IBES adjusted figure on both sides of the Earnings Surprise and SUE calculations is consistent with the 'analyst consensus vs. actual' framing in the Part 1 signal definition.

In [3]:
def wrds_get(path, params, cache_path=None):
    """Hit a WRDS REST endpoint and return the parsed JSON. Caches to disk if cache_path given."""
    if cache_path and cache_path.exists():
        return json.loads(cache_path.read_text())
    headers = {
        'Accept': 'application/json',
        'Authorization': f'Token {WRDS_TOKEN}',
    }
    r = requests.get(f'{WRDS_BASE}/{path}/', headers=headers, params=params, timeout=30)
    r.raise_for_status()
    payload = r.json()
    if cache_path:
        cache_path.write_text(json.dumps(payload, indent=2))
    return payload

# 2.1 Quarterly EPS actuals (USD), 2024 onward
actuals = wrds_get(
    'ibes.actu_epsus',
    params={'ticker': 'JNJ', 'pdicity': 'QTR', 'pends__gte': '2024-01-01'},
    cache_path=DATA_DIR / 'ibes_jnj_quarterly_actuals.json',
)
actu_df = pd.DataFrame(actuals['results'])[['pends', 'anndats', 'pdicity', 'value', 'curr_act']]
# WRDS returns both ANN and QTR rows even when pdicity=QTR is requested. Force-filter.
actu_df = actu_df[actu_df['pdicity'] == 'QTR'].sort_values('pends').reset_index(drop=True)
print(actu_df)

        pends     anndats pdicity  value curr_act
0  2024-03-31  2024-04-16     QTR   2.71      USD
1  2024-06-30  2024-07-17     QTR   2.82      USD
2  2024-09-30  2024-10-15     QTR   2.42      USD
3  2024-12-31  2025-01-22     QTR   2.04      USD
4  2025-03-31  2025-04-15     QTR   2.77      USD
5  2025-06-30  2025-07-16     QTR   2.77      USD
6  2025-09-30  2025-10-14     QTR   2.80      USD
7  2025-12-31  2026-01-21     QTR   2.46      USD


In [4]:
# 2.2 FY 2025 annual consensus history
consensus = wrds_get(
    'ibes.statsum_epsus',
    params={'ticker': 'JNJ', 'fpedats': '2025-12-31', 'measure': 'EPS', 'fiscalp': 'ANN', 'limit': 100},
    cache_path=DATA_DIR / 'ibes_jnj_fy2025_annual_consensus.json',
)
cons_df = pd.DataFrame(consensus['results'])[['statpers', 'meanest', 'medest', 'numest', 'stdev', 'curcode']]
cons_df['statpers'] = pd.to_datetime(cons_df['statpers'])
cons_df = cons_df.sort_values('statpers').reset_index(drop=True)
print(cons_df.tail(10))

     statpers  meanest  medest  numest  stdev curcode
62 2025-09-18    10.86   10.85    22.0   0.07     USD
63 2025-09-18     2.57    2.59    16.0   0.06     USD
64 2025-10-16     2.54    2.54    16.0   0.03     USD
65 2025-10-16    10.87   10.86    22.0   0.03     USD
66 2025-11-20     2.53    2.53    16.0   0.03     USD
67 2025-11-20    10.86   10.86    22.0   0.04     USD
68 2025-12-18     2.53    2.53    17.0   0.03     USD
69 2025-12-18    10.87   10.87    23.0   0.04     USD
70 2026-01-15     2.44    2.44    12.0   0.03     USD
71 2026-01-15    10.77   10.78    13.0   0.03     USD


In [5]:
# 2.3 Most recent statpers before the 21 January 2026 announcement
latest_consensus_row = cons_df[cons_df['statpers'] < Q4_2025_ANNOUNCEMENT].iloc[-1]
fy_2025_consensus = float(latest_consensus_row['meanest'])
print(f"Latest pre-announcement annual consensus: ${fy_2025_consensus:.2f} "
      f"(statpers={latest_consensus_row['statpers'].date()}, n_est={int(latest_consensus_row['numest'])})")

# Q1-Q3 2025 actuals (USD) sum
q123_2025 = actu_df[actu_df['pends'].isin(['2025-03-31', '2025-06-30', '2025-09-30'])]
q123_sum = float(q123_2025['value'].astype(float).sum())
print(f"Q1+Q2+Q3 2025 actuals (USD): ${q123_sum:.2f}")

# Q4 2025 actual is available directly from IBES
Q4_2025_ACTUAL = float(
    actu_df.loc[(actu_df['pends'] == '2025-12-31') & (actu_df['pdicity'] == 'QTR'), 'value'].iloc[0]
)
Q4_2024_ACTUAL = float(
    actu_df.loc[(actu_df['pends'] == '2024-12-31') & (actu_df['pdicity'] == 'QTR'), 'value'].iloc[0]
)
Q4_2025_CONSENSUS = fy_2025_consensus - q123_sum

print(f"Q4 2025 actual (IBES):        ${Q4_2025_ACTUAL:.2f}")
print(f"Q4 2025 implied consensus:    ${Q4_2025_CONSENSUS:.2f}")
print(f"Q4 2024 actual (IBES):        ${Q4_2024_ACTUAL:.2f}")

Latest pre-announcement annual consensus: $10.77 (statpers=2026-01-15, n_est=13)
Q1+Q2+Q3 2025 actuals (USD): $8.34
Q4 2025 actual (IBES):        $2.46
Q4 2025 implied consensus:    $2.43
Q4 2024 actual (IBES):        $2.04


## 3. Yahoo! Finance prices

Two series: JNJ NYSE common stock (USD) and S&P 500 (USD, market benchmark for Momentum). Window covers April 2024 through April 2026 to support the 12-month Volatility window (ending 30 Apr 2026) and the 12-month Momentum window (ending 31 Mar 2026).

In [6]:
def get_prices(ticker, start='2024-04-01', end='2026-05-01'):
    fname = DATA_DIR / f"prices_{ticker.replace('^', '').replace('.', '_')}.csv"
    if fname.exists():
        return pd.read_csv(fname, parse_dates=['Date'], index_col='Date')
    df = yf.download(ticker, start=start, end=end, auto_adjust=False, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] for c in df.columns]
    df.to_csv(fname)
    return df

jnj = get_prices('JNJ')
spx = get_prices('^GSPC')

print('JNJ           last close: $', jnj["Adj Close"].iloc[-1])
print('S&P 500       last close: ', spx["Adj Close"].iloc[-1])

JNJ           last close: $ 229.8500061035156
S&P 500       last close:  7209.009765625


## 4. Financial line items from the 10-K

Source: Johnson & Johnson Form 10-K for FY 2025, filed with the SEC on 11 February 2026 (accession 0000200406-26-000016). Line items below are in USD millions for FY 2025 (year ended 28 December 2025) and FY 2024 (year ended 31 December 2024). Note that FY 2024 net income of $14,066 million is depressed by approximately $7 billion in talc-related litigation charges; FY 2025 net income of $26,804 million reflects the absence of comparable charges and is closer to the company's underlying earnings power.

FY 2024 figures are required for the F-Score year-over-year comparators (ROA, leverage, current ratio, gross margin, asset turnover).

In [7]:
# Source: Johnson & Johnson Form 10-K for FY 2025, filed 11 February 2026 (SEC accession 0000200406-26-000016).
# Filing URL: https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=0000200406&type=10-K
# Values were extracted via stockanalysis.com (which aggregates SEC EDGAR XBRL data) and
# cross-referenced against the 21 January 2026 8-K Q4/FY 2025 release headline figures
# (sales $94.2 billion, FY reported EPS $11.03, FY adjusted EPS $10.79). Before final
# report submission, each line item below should be reconciled directly against the 10-K PDF.
#
# Section references in the consolidated financial statements of the 10-K:
#   - Revenue, COGS, Net Income     -> Consolidated Statements of Earnings
#   - Total Assets, Current Assets,
#     Total Liabilities, Current Liabilities,
#     Long-Term Debt, Total Equity  -> Consolidated Balance Sheets
#   - CFO, LT Debt Issued/Repaid,
#     Common Stock Issued/Repurchased -> Consolidated Statements of Cash Flows (financing section)
#
# All values in USD millions unless flagged otherwise.
fin = {
    'revenue':            {'2025':  94193, '2024':  88821},
    'cogs':               {'2025':  30256, '2024':  27471},
    'gross_profit':       {'2025':  63937, '2024':  61350},
    'net_income':         {'2025':  26804, '2024':  14066},   # FY 2024 depressed by talc litigation
    'cfo':                {'2025':  24530, '2024':  24266},
    'total_assets':       {'2025': 199210, '2024': 180104},
    'total_current_assets':{'2025': 55624, '2024':  55893},
    'total_liabilities':  {'2025': 117666, '2024': 108614},
    'total_current_liab': {'2025':  54126, '2024':  50321},
    'long_term_debt':     {'2025':  39438, '2024':  30651},
    'total_equity':       {'2025':  81544, '2024':  71490},
    'lt_debt_issued':     {'2025':   9138, '2024':   6660},
    'lt_debt_repaid':     {'2025':   1757, '2024':   1453},
    'equity_issued':      {'2025':   3418, '2024':    838},   # employee stock plans (ESPP, options)
    'shares_repurchased': {'2025':   5953, '2024':   2432},
}

# Total shares outstanding at fiscal year end 2025.
# Source: 10-K cover page reports approximately 2.407 billion shares of common stock outstanding.
TOTAL_SHARES_OUTSTANDING = 2_407_000_000

pd.DataFrame(fin).T

,2025,2024
revenue,94193,88821
cogs,30256,27471
gross_profit,63937,61350
net_income,26804,14066
cfo,24530,24266
total_assets,199210,180104
total_current_assets,55624,55893
total_liabilities,117666,108614
total_current_liab,54126,50321
long_term_debt,39438,30651


## 5. Insider transactions (Form 4)

All Johnson & Johnson Form 4 filings between 30 January 2026 and 30 April 2026 (the trailing three-month window ending 30 April 2026) were reviewed via SEC EDGAR for CIK 0000200406. Per the Part 1 signal definition, only code-P (open-market purchase) and code-S (open-market sale) transactions involving CEO Joaquin Duato, CFO Joseph Wolk, COO equivalents, or Board Directors are counted; code-A (compensation grants), code-F (tax withholding on vesting), code-G (gift), and code-M (option exercise without sale) are excluded.

Form 4 filings in the window by individuals in scope:
- Eugene A. Woods (Director): 154.869 Deferred Share Units granted 12 March 2026 (code A/G, compensation under the company's Deferred Fee Plan for Directors). Excluded from the signal.
- Daniel E. Pinto (Director): 1,712 cash-settled Deferred Share Units granted 23 April 2026 (code A/G, compensation). Excluded from the signal.

Form 4 filings in the window by individuals out of scope (per the assignment's CEO/CFO/COO/Director restriction): Robert J. Decker (VP Corporate Controller), Timothy Schmid (EVP), James D. Swanson (Executive), Kathryn E. Wengel (EVP, Chief Risk Officer), Kristen Mulholland (EVP, Chief HR Officer). Johnson & Johnson does not currently have a single-titled Chief Operating Officer following the 2018 reorganization.

No CEO Joaquin Duato or CFO Joseph Wolk Form 4 filings appear in the window. With no open-market P or S transactions among the in-scope insiders, the signal takes value zero per the assignment's "zero if missing" rule.

In [8]:
# Per the review above, no in-scope insider executed an open-market purchase or sale
# in the trailing three-month window. Signal is set to zero.
shares_sold_open_market = 0
shares_purchased_open_market = 0

## 6. Signal computations

All ten signals from the Part 1 specification. F-Score is integer (0-9); the rest are decimal.

In [9]:
# 6.1 Insider Selling
inssell = (shares_sold_open_market - shares_purchased_open_market) / TOTAL_SHARES_OUTSTANDING
print(f"Insider Selling: {inssell:.10f}")

Insider Selling: 0.0000000000


In [10]:
# 6.2 Book-to-Market
jnj_close_cutoff = float(jnj.loc[jnj.index == CUTOFF, 'Adj Close'].iloc[0])
market_cap = jnj_close_cutoff * TOTAL_SHARES_OUTSTANDING
btom = (fin['total_equity']['2025'] * 1e6) / market_cap
print(f"JNJ close on {CUTOFF.date()}: ${jnj_close_cutoff:.2f}")
print(f"Market cap: ${market_cap/1e9:.1f} bn")
print(f"Book equity: ${fin['total_equity']['2025']/1e3:.1f} bn")
print(f"Book-to-Market: {btom:.4f}")

JNJ close on 2026-04-30: $229.85
Market cap: $553.2 bn
Book equity: $81.5 bn
Book-to-Market: 0.1474


In [11]:
# 6.3 Momentum (USD basis with S&P 500 as the market benchmark)
def closest_close(df, target):
    sub = df[df.index <= target]
    return float(sub['Adj Close'].iloc[-1])

jnj_end = closest_close(jnj, MOMENTUM_WINDOW_END)
jnj_start = closest_close(jnj, MOMENTUM_WINDOW_START)
spx_end = closest_close(spx, MOMENTUM_WINDOW_END)
spx_start = closest_close(spx, MOMENTUM_WINDOW_START)

jnj_ret = jnj_end / jnj_start - 1
spx_ret = spx_end / spx_start - 1
momentum = jnj_ret - spx_ret

print(f"JNJ cum return ({MOMENTUM_WINDOW_START.date()} -> {MOMENTUM_WINDOW_END.date()}): {jnj_ret:.4f}")
print(f"S&P 500 cum return: {spx_ret:.4f}")
print(f"Momentum (market-adjusted): {momentum:.4f}")

JNJ cum return (2025-03-31 -> 2026-03-31): 0.5150
S&P 500 cum return: 0.1633
Momentum (market-adjusted): 0.3517


In [12]:
# 6.4 Gross Profitability
grossprofit = (fin['revenue']['2025'] - fin['cogs']['2025']) / fin['total_assets']['2025']
print(f"Gross Profitability: {grossprofit:.4f}")

Gross Profitability: 0.3210


In [13]:
# 6.5 Earnings Surprise
jnj_5d_before = float(jnj.loc[jnj.index < Q4_2025_ANNOUNCEMENT, 'Adj Close'].iloc[-5])
jnj_5d_date = jnj.loc[jnj.index < Q4_2025_ANNOUNCEMENT].index[-5]
earnsurprise = (Q4_2025_ACTUAL - Q4_2025_CONSENSUS) / jnj_5d_before
print(f"JNJ price 5 trading days before {Q4_2025_ANNOUNCEMENT.date()}: ${jnj_5d_before:.2f} (on {jnj_5d_date.date()})")
print(f"Q4 2025 actual - implied consensus: ${Q4_2025_ACTUAL - Q4_2025_CONSENSUS:.2f}")
print(f"Earnings Surprise: {earnsurprise:.4f}")

JNJ price 5 trading days before 2026-01-21: $212.52 (on 2026-01-13)
Q4 2025 actual - implied consensus: $0.03
Earnings Surprise: 0.0001


In [14]:
# 6.6 SUE
sue = (Q4_2025_ACTUAL - Q4_2024_ACTUAL) / jnj_5d_before
print(f"Q4 2025 actual - Q4 2024 actual: ${Q4_2025_ACTUAL - Q4_2024_ACTUAL:.2f}")
print(f"SUE: {sue:.4f}")

Q4 2025 actual - Q4 2024 actual: $0.42
SUE: 0.0020


In [15]:
# 6.7 F-Score (Piotroski 9-criterion)
checks = {}
checks['ROA > 0'] = int(fin['net_income']['2025'] / fin['total_assets']['2025'] > 0)
checks['CFO > 0'] = int(fin['cfo']['2025'] > 0)
checks['DROA > 0'] = int(
    fin['net_income']['2025'] / fin['total_assets']['2025']
    > fin['net_income']['2024'] / fin['total_assets']['2024']
)
checks['CFO > NI'] = int(fin['cfo']['2025'] > fin['net_income']['2025'])
checks['DLeverage < 0'] = int(
    fin['long_term_debt']['2025'] / fin['total_assets']['2025']
    < fin['long_term_debt']['2024'] / fin['total_assets']['2024']
)
checks['DCurrent ratio > 0'] = int(
    fin['total_current_assets']['2025'] / fin['total_current_liab']['2025']
    > fin['total_current_assets']['2024'] / fin['total_current_liab']['2024']
)
checks['No equity issuance'] = int(fin['equity_issued']['2025'] == 0)
checks['DGross margin > 0'] = int(
    (fin['revenue']['2025'] - fin['cogs']['2025']) / fin['revenue']['2025']
    > (fin['revenue']['2024'] - fin['cogs']['2024']) / fin['revenue']['2024']
)
checks['DAsset turnover > 0'] = int(
    fin['revenue']['2025'] / fin['total_assets']['2025']
    > fin['revenue']['2024'] / fin['total_assets']['2024']
)
fscore = sum(checks.values())
for k, v in checks.items():
    print(f"  {k:25s} {v}")
print(f"F-Score: {fscore}")

  ROA > 0                   1
  CFO > 0                   1
  DROA > 0                  1
  CFO > NI                  0
  DLeverage < 0             0
  DCurrent ratio > 0        0
  No equity issuance        0
  DGross margin > 0         0
  DAsset turnover > 0       0
F-Score: 3


In [16]:
# 6.8 External Financing
xfin_num = (
    fin['lt_debt_issued']['2025'] + fin['equity_issued']['2025']
    - fin['lt_debt_repaid']['2025'] - fin['shares_repurchased']['2025']
)
xfin = xfin_num / fin['total_assets']['2025']
print(f"Net financing (USD m): {xfin_num:,}")
print(f"External Financing: {xfin:.4f}")

Net financing (USD m): 4,846
External Financing: 0.0243


In [17]:
# 6.9 Volatility (JNJ NYSE common, simple daily returns, 12 months ending 30 Apr 2026)
vol_window = jnj[(jnj.index > VOL_WINDOW_START) & (jnj.index <= VOL_WINDOW_END)]
daily_returns = vol_window['Adj Close'].pct_change().dropna()
volatility = float(daily_returns.std())
print(f"N daily returns: {len(daily_returns)}")
print(f"Volatility: {volatility:.4f}")

N daily returns: 250
Volatility: 0.0106


In [18]:
# 6.10 Accruals (FY annual)
accruals = (fin['net_income']['2025'] - fin['cfo']['2025']) / fin['total_assets']['2025']
print(f"Accruals: {accruals:.4f}")

Accruals: 0.0114


## 7. Output table

Columns: Signal | Value | Source | As-Of.

In [19]:
table = pd.DataFrame([
    {'Signal': 'Insider Selling',     'Value': inssell,
     'Source': 'SEC Form 4 filings',
     'As-Of': '2026-04-30'},
    {'Signal': 'Book-to-Market',      'Value': btom,
     'Source': '10-K for the fiscal year ending 12/28/25; Yahoo! Finance',
     'As-Of': '2026-04-30'},
    {'Signal': 'Momentum',            'Value': momentum,
     'Source': 'Yahoo! Finance',
     'As-Of': '2026-03-31'},
    {'Signal': 'Gross Profitability', 'Value': grossprofit,
     'Source': '10-K for the fiscal year ending 12/28/25',
     'As-Of': '2025-12-28'},
    {'Signal': 'Earnings Surprise',   'Value': earnsurprise,
     'Source': '8-K earnings release filed 1/21/26; WRDS IBES; Yahoo! Finance',
     'As-Of': '2026-01-21'},
    {'Signal': 'SUE',                 'Value': sue,
     'Source': '8-K earnings release filed 1/21/26; WRDS IBES; Yahoo! Finance',
     'As-Of': '2026-01-21'},
    {'Signal': 'F-Score',             'Value': fscore,
     'Source': '10-K for the fiscal year ending 12/28/25',
     'As-Of': '2025-12-28'},
    {'Signal': 'External Financing',  'Value': xfin,
     'Source': '10-K for the fiscal year ending 12/28/25',
     'As-Of': '2025-12-28'},
    {'Signal': 'Volatility',          'Value': volatility,
     'Source': 'Yahoo! Finance',
     'As-Of': '2026-04-30'},
    {'Signal': 'Accruals',            'Value': accruals,
     'Source': '10-K for the fiscal year ending 12/28/25',
     'As-Of': '2025-12-28'},
])

def fmt(row):
    if row['Signal'] == 'F-Score':
        return int(row['Value'])
    if row['Signal'] == 'Insider Selling':
        return '0' if row['Value'] == 0 else f"{row['Value']:.7f}"
    return f"{row['Value']:.4f}"

out = table.copy()
out['Value'] = table.apply(fmt, axis=1)
print(out.to_string(index=False))

out.to_csv(OUTPUT_DIR / 'jnj_signal_table.csv', index=False)
print('Wrote outputs/jnj_signal_table.csv')

             Signal  Value                                                        Source      As-Of
    Insider Selling      0                                            SEC Form 4 filings 2026-04-30
     Book-to-Market 0.1474      10-K for the fiscal year ending 12/28/25; Yahoo! Finance 2026-04-30
           Momentum 0.3517                                                Yahoo! Finance 2026-03-31
Gross Profitability 0.3210                      10-K for the fiscal year ending 12/28/25 2025-12-28
  Earnings Surprise 0.0001 8-K earnings release filed 1/21/26; WRDS IBES; Yahoo! Finance 2026-01-21
                SUE 0.0020 8-K earnings release filed 1/21/26; WRDS IBES; Yahoo! Finance 2026-01-21
            F-Score      3                      10-K for the fiscal year ending 12/28/25 2025-12-28
 External Financing 0.0243                      10-K for the fiscal year ending 12/28/25 2025-12-28
         Volatility 0.0106                                                Yahoo! Finance 2026-04-30


## 8. Notes on methodology

Johnson & Johnson is a US-domestic NYSE-listed issuer. The disclosure regime (10-K, 8-K, 10-Q, Form 4) and reporting standard (US GAAP, USD) are standard. Three items warrant a brief note:

1. **No Chief Operating Officer.** Following the 2018 reorganization, Johnson & Johnson does not have a single-titled COO. The Form 4 review for insider selling therefore restricts to the CEO (Joaquin Duato), CFO (Joseph Wolk), and Board Directors. Divisional Executive Vice Presidents are out of scope per the assignment's CEO/CFO/COO/Director restriction.
2. **Adjusted versus reported EPS for Earnings Surprise.** Q4 2025 reported diluted EPS was $2.10 and adjusted EPS was $2.46. WRDS IBES tracks the adjusted series on both the actual and consensus sides, which is the standard street basis. The implied Q4 2025 consensus is recovered from the latest pre-announcement IBES annual mean (statpers 15 January 2026) less the IBES Q1-Q3 2025 actuals; the Q4 2024 actual ($2.04) is taken directly from IBES.
3. **Conventions for Accruals and External Financing.** Accruals is computed on an FY annual basis rather than the literal "most recent fiscal quarter" wording in Part 1. External Financing includes long-term debt activity only. JNJ's positive Accruals value is mechanically driven by FY 2024 net income being depressed by approximately $7 billion in talc litigation charges; on a talc-normalized basis the value would be closer to zero.